In [1]:
import logging
import sys
import os

# Add the project root directory to Python path using our project initialization module
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..')))
# Configure logging to show INFO level messages
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
os.listdir()

['.gitignore',
 '.python-version',
 '.venv',
 'app',
 'data',
 'demo.py',
 'example',
 'main.py',
 'pyproject.toml',
 'README.md',
 'uv.lock',
 'uv.toml']

In [2]:
# 测试openai
from openai import OpenAI

client = OpenAI(base_url='http://172.16.2.158/api/llm2/v1', api_key='sk-1223swassqqqq33')

# Example API call to completion model (text generation)
response = client.chat.completions.create(
    model='Qwen3-32B',
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"}
    ],
    extra_body={
        "enable_thinking": False
    },
    max_tokens=150
)

print(response.choices[0].message.content)

2025-09-20 08:31:02,172 - httpx - INFO - HTTP Request: POST http://172.16.2.158/api/llm2/v1/chat/completions "HTTP/1.1 200 OK"


The **Los Angeles Dodgers** won the **2020 World Series**. They defeated the **Tampa Bay Rays** in **6 games** (4–2) to claim their **seventh World Series title in franchise history** and their first since 1988. The 2020 season was shortened to 60 games due to the COVID-19 pandemic, but the playoffs still took place. 

Key highlights include:

- **Clayton Kershaw** won the **World Series MVP**.
- It was the first World Series since 1903 to be played without fans in attendance due to the pandemic.
- The Dodgers swept the NLCS in three games, and the World


In [5]:
from openai import OpenAI
embedding_client = OpenAI(base_url="http://172.16.2.158/api/nlp-model/v1", api_key="sk-1234")
response = embedding_client.embeddings.create(
    input=["hello world",'你好世界'],
    model="bge-m3"  # 例如 "text-embedding-ada-002"
)

print(len([item.embedding for item in response.data]))
# 获取嵌入向量
embedding = response.data[0].embedding
print(f"嵌入向量维度: {len(embedding)}")
print(f"前10个元素: {embedding[:10]}")

1
嵌入向量维度: 1024
前10个元素: [-0.04022522, 0.0369581, -0.029007481, 0.016184513, -0.035749454, -0.040791772, -0.05521997, -0.041018393, 0.0032623974, 0.001988835]


In [6]:
# 测试openai
from openai import OpenAI

client = OpenAI(base_url='http://172.16.2.158/api/llm2/v1', api_key='sk-1223swassqqqq33')

# Example API call to completion model (text generation)
response = client.completions.create(
    model='Qwen3-32B',
    prompt='你好，世界! //no_thinking',
    max_tokens=150
)

print(response.choices[0].text)

2025-09-20 08:33:00,921 - httpx - INFO - HTTP Request: POST http://172.16.2.158/api/llm2/v1/completions "HTTP/1.1 200 OK"



你好！😊 有什么我可以帮助你的吗？是想聊聊天，还是有具体的问题需要解决呢？


In [27]:
from datetime import datetime
import json
import os
import random
from typing import List, Union
# 或者如果需要更具体的类型提示：
from openai.types.chat import ChatCompletionToolUnionParam

# 模拟用户问题
USER_QUESTION = "新加坡天气咋样"
# 定义工具列表
tools: List[ChatCompletionToolUnionParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "当你想查询指定城市的天气时非常有用。",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "城市或县区，比如新加坡、纽约等。",
                    }
                },
                "required": ["location"],
            },
        },
    },
]

# 模拟天气查询工具
def get_current_weather(arguments):
    weather_conditions = ["晴天", "多云", "雨天"]
    random_weather = random.choice(weather_conditions)
    location = arguments["location"]
    return f"{location}今天是{random_weather}。"

# 封装模型响应函数
def get_response(messages):
    completion = client.chat.completions.create(
        model="Qwen3-32B",
        messages=messages,
        tools=tools,
    )
    return completion


In [28]:
messages = [{"role": "user", "content": USER_QUESTION}]
response = get_response(messages)
assistant_output = response.choices[0].message
if assistant_output.content is None:
    assistant_output.content = ""
messages.append(assistant_output)
# 如果不需要调用工具，直接输出内容
if assistant_output.tool_calls is None:
    print(f"无需调用天气查询工具，直接回复：{assistant_output.content}")
else:
    # 进入工具调用循环
    while assistant_output.tool_calls is not None:
        tool_call = assistant_output.tool_calls[0]
        tool_call_id = tool_call.id
        func_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"正在调用工具 [{func_name}]，参数：{arguments}")
        # 执行工具
        tool_result = get_current_weather(arguments)
        # 构造工具返回信息
        tool_message = {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "content": tool_result,  # 保持原始工具输出
        }
        print(f"工具返回：{tool_message['content']}")
        messages.append(tool_message)
        # 再次调用模型，获取总结后的自然语言回复
        response = get_response(messages)
        assistant_output = response.choices[0].message
        if assistant_output.content is None:
            assistant_output.content = ""
        messages.append(assistant_output)
    print(f"助手最终回复：{assistant_output.content}")

2025-09-20 09:16:04,075 - httpx - INFO - HTTP Request: POST http://172.16.2.158/api/llm2/v1/chat/completions "HTTP/1.1 200 OK"


正在调用工具 [get_current_weather]，参数：{'location': '新加坡'}
工具返回：新加坡今天是雨天。


2025-09-20 09:16:05,666 - httpx - INFO - HTTP Request: POST http://172.16.2.158/api/llm2/v1/chat/completions "HTTP/1.1 200 OK"


助手最终回复：新加坡今天是雨天，记得带伞出门哦！
